In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

In [4]:
def llm(prompt):
    response = openai_client.responses.create(
        model="openai/gpt-oss-120b:free",
        input=prompt
    )
    return response.output_text

In [5]:
llm("What is the capital of germany")

'The capital of Germany is **Berlin**.'

In [6]:
question = "where is the nearest shawarma joint from tunfure"
answer = llm(question)
print(answer)

I’m sorry, but I can’t help with that.


In [7]:
question = "where is the nearest shawarma joint from tunfure"


In [8]:
# adding a context

context = f"""
Tunfure is a province in Gombe State Nigeria, it has the most places for a night joint eatries like Suya and Shawarma.

the closest place to get sawarma and suya is at City Star Restaurant at Orji quarters
"""

In [9]:
prompt = f"""
your task is to answer the question base on the provided context

use the context to find relevant and provide accurate answers, if you cant find relevant answers reply with 'i dont know'

question:
{question}

context:
{context}
"""

In [10]:
answer = llm(prompt)
print(answer)

The nearest place to get shawarma (and suya) from Tunfure is **City Star Restaurant in Orji Quarters**.


## The course FAQ Dataset

Before we build the RAG pipeline, let's get familiar with the data we'll use as our knowledge base.

We run these courses every year, and students keep asking the same questions in Slack. We collected those into an FAQ so people can find answers before asking. Some courses have run for five cohorts, so the FAQ grows large and searching it by hand gets tedious. That's exactly the problem our RAG system will solve.

The FAQ data is available as JSON from the DataTalks.Club website. I maintain that site, so I made the data available at a JSON endpoint we can fetch directly.

Let's fetch it:

In [11]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

This returns a list of courses. Each course has a `path` field that
points to its FAQ data.

Let's fetch all the FAQ documents from all
courses:

In [12]:
from httpcore import request
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

Each entry has:

- id - unique identifier
- course - course slug (e.g., machine-learning-zoomcamp)
- section - which section of the course
- question - the FAQ question
- answer - the FAQ answer

Let's look at one:

In [13]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

Each course has a slug - a short identifier used in URLs. For example, machine-learning-zoomcamp, data-engineering-zoomcamp, etc. We'll use these slugs for filtering in search.

## Using this data

In the RAG pipeline, this dataset is our knowledge base:

1. We index all the documents (the search step)
2. When a student asks a question, we search the index
3. The search returns the most relevant FAQ entries
4. We give those entries to the LLM as context
5. The LLM generates an answer based on the context

The `question` and `answer` fields contain the text we'll search
through. The `course` field lets us filter by course. For example, if a
student asks about the data engineering course, we skip results from
the ML course. The `section` field helps with ranking - knowing which
part of the course a question belongs to is useful context.

## A note on data preparation

In our case, the data is already prepared. I maintain this FAQ website
and made sure the data comes back in a convenient JSON format. So we
don't need to do much to get it ready. By the way, I cleaned a lot of
this data with the help of an LLM. That's a handy use case on its own.

Don't let that fool you. In reality, data preparation is often the most
time-consuming part of building a RAG system. You may need to scrape
websites, parse PDFs, and clean and chunk documents. That work isn't
visible here, but I did plenty of it ahead of time.

We keep the focus on the GenAI side in this course. In your own
projects, expect to spend real time on data preparation before you get
to this point.

In the next section, we'll build the search index.

In [14]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [15]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

We get back 5 results from the LLM Zoomcamp FAQ. The best match is the FAQ entry "I just discovered the course. Can I still join?" which is exactly what we need.

Here are all the questions:



In [16]:
[doc['question'] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

## Wrapping it in a function
---
Let's wrap the search in a search function - the first component of our RAG pipeline:

In [ ]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5 }
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        question,
        boost_dict = boost_dict,
        filter_dict = filter_dict,
        num_results = 5

    )

In [18]:
search_results = search(question)

In [19]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

## Building the Prompt

The LLM doesn't see our documents unless we pass them in. So we need to build a prompt that includes the user's question and the search results.

When we build AI systems, we usually split the prompt into two parts:

Instructions (also called the system prompt): this tells the LLM how to behave. It never changes, so it's the same for every request.

User prompt: this changes with every request. It carries the actual question and the retrieved context.

We split them because the instructions are fixed and the user prompt is not. Keeping them apart makes the fixed part easy to reuse and the changing part easy to build fresh each time.

Instructions
The instructions tell the LLM its role and how to answer:



In [20]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [21]:
# This is what grounds the answer in our data and reduces hallucinations.

# The user prompt template
# The user prompt template has placeholders for the question and the context:

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""
# Building the context
# The context is a formatted string with all the search results:

In [22]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

Each document becomes a block with the section, question, and answer. This format makes it easy for the LLM to read. We turned a list of dictionaries into one string. It's a small preprocessing step before we send the data to the LLM.

---

## Building the prompt
Now we combine the question with the context into the user prompt:



In [23]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )

    return prompt.strip()

In [24]:
prompt = build_prompt(question, search_results)
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

## Exploring the response

---

The response is a Pydantic object. The answer is in response.output - a list of output items.

The first one is the message:

In [25]:
response = openai_client.responses.create(
    model="openai/gpt-oss-120b:free",
    input=prompt
)

In [26]:
response.output[0]

ResponseReasoningItem(id='rs_tmp_pgfa7zpmpxs', summary=[], type='reasoning', content=[Content(text='The answer: yes you can join now but need to submit project before deadline for certificate.', type='reasoning_text')], encrypted_content=None, status='completed', format='unknown')

In [27]:
# The message has a content list, and the text is in the first item:

response.output[0].content[0].text

'The answer: yes you can join now but need to submit project before deadline for certificate.'

In [28]:
# That's quite a journey to reach one string.
# The shortcut spares us all of it:
response.output_text

'**Yes – you can still join the course.**  \nJust keep in mind that if you want a certificate you’ll need to submit your capstone project while the cohort is still open for submissions (and later complete the required peer‑reviews). Otherwise you can start learning right away.'

In [29]:
response.usage

ResponseUsage(input_tokens=541, input_tokens_details=InputTokensDetails(cached_tokens=64), output_tokens=85, output_tokens_details=OutputTokensDetails(reasoning_tokens=23), total_tokens=626, cost=0, is_byok=False, cost_details={'upstream_inference_cost': 0, 'upstream_inference_input_cost': 0, 'upstream_inference_output_cost': 0})

In [30]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.0007882500000000001

This particular request costs a fraction of a cent. Even a full RAG query with a long prompt stays under $0.01. We need to send a lot of queries to even spend one cent. These models are cheap to play with.

The usage object also reports cached input tokens. Those are billed at a lower rate when the same prompt prefix repeats.

## Message history

---

Previously we sent only one string as input and got back a response. In practice, you typically send a message history - a list of messages where each message has a role.

Think of a ChatGPT conversation. It starts with a hidden system prompt that tells the LLM how to behave, one you never see. After that, your messages and the LLM's replies alternate. The LLM has no memory of its own, so it needs the full history passed in to continue the conversation.

We won't build a multi-turn chat here. But we still use this message format to separate our instructions from the user prompt.

We send two messages:

- developer - system-level instructions (how the LLM should behave)
- user - the actual prompt with the question and context


In [31]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model = "openai/gpt-oss-120b:free",
    input = message_history
)

This separates the fixed instructions from the user prompt, which changes every request.

OpenAI accepts both developer and system for the instruction role. There may be some difference between them, but in practice I don't see it change the result either way. We use developer in this course.

## The LLM function

---

We can now put this together into an updated llm function.

It now takes both instructions and the user prompt:



In [32]:
def llm(instructions, user_prompt, model="openai/gpt-oss-120b:free"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model = model,
        input = message_history
    )

    return response.output_text

## Full RAG
---

With search, the prompt, and the LLM ready, we wire them together:



In [33]:
def rag(query, model="openai/gpt-oss-120b:free"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)

    return answer

```mermaid
flowchart TD
    U([User])

    APP[Application]

    DB[(DB)]
    DOCS[[D1 ... D5]]

    PROMPT[Build Prompt<br/>Question + Context]

    LLM[LLM]

    ANSWER([Answer])

    U -->|Question| APP

    APP -->|Query| DB
    DB -->|Retrieved Data| DOCS
    DOCS --> APP

    APP --> PROMPT
    PROMPT --> LLM

    LLM --> ANSWER
    ANSWER --> U

```

In [35]:
answer = rag("I just discovered this course can i stll join it?")
answer

'Yes—you can still join the course. Just sign up and start learning; if you want a certificate, be sure to submit your project while the submission window is open.'

In [37]:
rag("can i get a certificate after the course kicks off?")

'Yes—if you enroll in the live cohort (i.e., while the course is running), you can still earn a certificate. You’ll need to submit your capstone project\u202f*before* the submission window closes and complete the required three peer‑reviews. Certificates are not offered for the self‑paced mode.'

Notice how the answers reference specific courses and sections. The LLM reads from our knowledge base before answering - that's how RAG works.

This approach is modular. You can swap the search backend, the prompt template, or the LLM model. Nothing else needs to change. Later when we replace minsearch with sqlitesearch, only the search function changes.